# 06 — Runner & Statistical Profiling

Covers:
- `Runner` — basic usage, strict vs. non-strict mode, parallel execution
- `enable_profiling=True` — automatic statistical profile attached to every `MonitoringReport`
- Profile structure for numeric, binary, and categorical columns
- Profiler standalone — calling `profile_columns()` directly outside the Runner
- Multiple configurations: drift-only, performance, mixed drift + fairness

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 500

# Reference window — stable production distribution
df_ref = pd.DataFrame(
    {
        "y_true": rng.integers(0, 2, size=n),
        "y_pred": rng.integers(0, 2, size=n),
        "y_pred_proba": rng.uniform(0.1, 0.9, size=n),
        "age": rng.normal(40, 8, size=n),
        "income": rng.normal(55_000, 12_000, size=n),
        "gender": rng.choice(["M", "F"], size=n),
        "region": rng.choice(["north", "south", "east", "west"], size=n),
    }
)

# Current window — age shifted, new region category, slight income drift
df_cur = pd.DataFrame(
    {
        "y_true": rng.integers(0, 2, size=n),
        "y_pred": rng.integers(0, 2, size=n),
        "y_pred_proba": rng.uniform(0.1, 0.9, size=n),
        "age": rng.normal(47, 10, size=n),  # mean shift +7 years
        "income": rng.normal(56_000, 13_500, size=n),  # slight scale increase
        "gender": rng.choice(["M", "F"], size=n),
        "region": rng.choice(["north", "south", "east", "west", "central"], size=n),  # new category
    }
)

print(f"Reference: {df_ref.shape}   Current: {df_cur.shape}")

Reference: (500, 7)   Current: (500, 7)


## 1. Runner — basic configuration

`Runner(strict=False)` — per-metric errors are collected in `report.errors` without aborting the run.  
`Runner(strict=True)` — raises on the first error (useful in CI).

In [2]:
from ayn_ml.core.schema import TabularSchema
from ayn_ml.core.spec import MetricSpec, MonitoringPlan
from ayn_ml.runner import Runner

schema = TabularSchema(
    label_col="y_true",
    prediction_col="y_pred",
    proba_col="y_pred_proba",
)

plan = MonitoringPlan(
    name="credit-scoring-v1",
    model_id="credit-model",
    model_version="1.0",
    data_schema=schema,
    metrics=[
        MetricSpec(name="ks_2samp", feature_name="age"),
        MetricSpec(name="ks_2samp", feature_name="income"),
        MetricSpec(name="wasserstein", feature_name="age"),
        MetricSpec(name="chisquare", feature_name="region"),
        MetricSpec(name="accuracy"),
        MetricSpec(name="auc"),
    ],
)

report = Runner(strict=False).run(plan, df_cur, reference=df_ref)

print(f"run_id      : {report.context.run_id}")
print(f"results     : {len(report.results)}")
print(f"errors      : {len(report.errors)}")
print(f"profile     : {report.profile}")  # None — profiling not enabled

run_id      : b382542c71154aabba9e6aea843ed999
results     : 6
errors      : 0
profile     : None


In [3]:
report.to_dataframe()

,metric_name,metric_type,feature_name,value,status,effect_size,effect_size_label,threshold,run_id,model_id,model_version,eval_timestamp
0,ks_2samp,None,age,0.000000,None,0.326000,ks_statistic,None,b382542c71154aabba9e6aea843ed999,credit-model,1.0,2026-05-27 02:05:05.720001+00:00
1,ks_2samp,None,income,0.002374,None,0.116000,ks_statistic,None,b382542c71154aabba9e6aea843ed999,credit-model,1.0,2026-05-27 02:05:05.720001+00:00
2,wasserstein,None,age,6.882620,None,NaN,None,None,b382542c71154aabba9e6aea843ed999,credit-model,1.0,2026-05-27 02:05:05.720001+00:00
3,chisquare,None,region,0.000000,None,0.324055,cramer_v,None,b382542c71154aabba9e6aea843ed999,credit-model,1.0,2026-05-27 02:05:05.720001+00:00
4,accuracy,None,None,0.480000,None,NaN,None,None,b382542c71154aabba9e6aea843ed999,credit-model,1.0,2026-05-27 02:05:05.720001+00:00
5,auc,None,None,0.512669,None,NaN,None,None,b382542c71154aabba9e6aea843ed999,credit-model,1.0,2026-05-27 02:05:05.720001+00:00


## 2. Runner with `enable_profiling=True`

Enabling profiling adds an automatic step after the metrics loop.  
Profiled columns = union of:
- `spec.feature_name` from all configured metrics
- schema target columns: `prediction_col`, `label_col`, `proba_col`

In [4]:
plan_with_profiling = plan.model_copy(update={"enable_profiling": True})

report_p = Runner(strict=False).run(plan_with_profiling, df_cur, reference=df_ref)

print("Profiled columns:", list(report_p.profile.keys()))

Colonnes profilées : ['age', 'income', 'region', 'y_pred', 'y_pred_proba', 'y_true']


In [5]:
# Profile of a numeric column
import json

print("age :")
print(json.dumps(report_p.profile["age"], indent=2))

age :
{
  "min": 20.74853315903198,
  "max": 75.45294394907638,
  "mean": 47.05250034143177,
  "std": 10.03759707455689,
  "p25": 39.729643456375996,
  "p50": 46.83276303947315,
  "p75": 53.582454274854236,
  "null_count": 0,
  "null_pct": 0.0
}


In [6]:
# Profile of a categorical column
print("region :")
print(json.dumps(report_p.profile["region"], indent=2))

region :
{
  "null_count": 0,
  "null_pct": 0.0,
  "n_unique": 5,
  "top_category": "south"
}


In [7]:
# Schema target columns — profiled automatically
print("y_pred (prediction_col) :")
print(json.dumps(report_p.profile["y_pred"], indent=2))
print("\ny_pred_proba (proba_col) :")
print(json.dumps(report_p.profile["y_pred_proba"], indent=2))

y_pred (prediction_col) :
{
  "min": 0.0,
  "max": 1.0,
  "mean": 0.502,
  "std": 0.5004967472301652,
  "p25": 0.0,
  "p50": 1.0,
  "p75": 1.0,
  "null_count": 0,
  "null_pct": 0.0
}

y_pred_proba (proba_col) :
{
  "min": 0.10051384295989899,
  "max": 0.8995271403101371,
  "mean": 0.49524260980828333,
  "std": 0.23471583188382175,
  "p25": 0.30076131745689405,
  "p50": 0.5035883028594937,
  "p75": 0.6905653380707146,
  "null_count": 0,
  "null_pct": 0.0
}


In [8]:
# Tabular profile comparison — reference vs current on numeric columns
from ayn_ml.metrics.tabular.profiler import profile_columns

numeric_cols = ["age", "income", "y_pred_proba"]
prof_ref = profile_columns(df_ref, numeric_cols, schema)
prof_cur = profile_columns(df_cur, numeric_cols, schema)

rows = []
for col in numeric_cols:
    for key in ["mean", "std", "p25", "p50", "p75"]:
        rows.append(
            {
                "column": col,
                "stat": key,
                "reference": round(prof_ref[col][key], 3),
                "current": round(prof_cur[col][key], 3),
            }
        )

pd.DataFrame(rows).pivot(index=["column", "stat"], columns=[], values=["reference", "current"])

reference    current
column       stat                      
age          mean     40.170     47.053
             std       8.080     10.038
             p25      35.001     39.730
             p50      40.251     46.833
             p75      45.375     53.582
income       mean  52875.998  56059.396
             std   12123.562  13196.018
             p25   44617.681  46801.605
             p50   53193.410  55611.065
             p75   61102.174  63932.967
y_pred_proba mean      0.499      0.495
             std       0.237      0.235
             p25       0.295      0.301
             p50       0.495      0.504
             p75       0.711      0.691

## 3. Multiple runner configurations

Same `Runner` instance, different plans — each `run()` produces a unique `run_id`.

In [9]:
runner = Runner(strict=False)

# Config A — drift only
plan_drift = MonitoringPlan(
    name="drift-only",
    model_id="credit-model",
    model_version="1.0",
    data_schema=schema,
    enable_profiling=True,
    metrics=[
        MetricSpec(name="ks_2samp", feature_name="age"),
        MetricSpec(name="ks_2samp", feature_name="income"),
        MetricSpec(name="wasserstein", feature_name="age"),
        MetricSpec(name="chisquare", feature_name="region"),
    ],
)

# Config B — performance only (no feature_name → only target cols are profiled)
plan_perf = MonitoringPlan(
    name="perf-only",
    model_id="credit-model",
    model_version="1.0",
    data_schema=schema,
    enable_profiling=True,
    metrics=[
        MetricSpec(name="accuracy"),
        MetricSpec(name="auc"),
        MetricSpec(name="f1"),
        MetricSpec(name="log_loss"),
    ],
)

# Config C — drift + performance + fairness
from ayn_ml.core.schema import TabularSchema

schema_fair = TabularSchema(
    label_col="y_true",
    prediction_col="y_pred",
    proba_col="y_pred_proba",
    protected_cols=["gender"],
)
plan_full = MonitoringPlan(
    name="full-monitoring",
    model_id="credit-model",
    model_version="1.0",
    data_schema=schema_fair,
    enable_profiling=True,
    metrics=[
        MetricSpec(name="ks_2samp", feature_name="age"),
        MetricSpec(name="psi", feature_name="income"),
        MetricSpec(name="chisquare", feature_name="region"),
        MetricSpec(name="accuracy"),
        MetricSpec(name="auc"),
        MetricSpec(name="demographic_parity", feature_name="gender"),
        MetricSpec(name="disparate_impact", feature_name="gender"),
    ],
)

for plan_cfg in (plan_drift, plan_perf, plan_full):
    r = runner.run(plan_cfg, df_cur, reference=df_ref)
    profiled = list(r.profile.keys()) if r.profile else []
    print(f"{plan_cfg.name:<20}  results={len(r.results)}  errors={len(r.errors)}  profiled={profiled}")

psi 'income': 3 current value(s) outside the reference range [1.843e+04, 8.997e+04]. PSI includes these values (union bins) but their contribution is sensitive to eps=0.0001 — PSI varies 1.7–3.1 for that bin alone at 30% out-of-range density. Cross-check with wasserstein or ks_2samp.


drift-only            results=4  errors=0  profiled=['age', 'income', 'region', 'y_pred', 'y_pred_proba', 'y_true']
perf-only             results=4  errors=0  profiled=['y_pred', 'y_pred_proba', 'y_true']
full-monitoring       results=7  errors=0  profiled=['age', 'gender', 'income', 'region', 'y_pred', 'y_pred_proba', 'y_true']


## 4. Same runner, two current windows — distinct `run_id`s

In [10]:
# Simulate two production batches arriving sequentially
df_batch1 = df_cur.iloc[:250].copy()
df_batch2 = df_cur.iloc[250:].copy()

r1 = runner.run(plan_drift, df_batch1, reference=df_ref)
r2 = runner.run(plan_drift, df_batch2, reference=df_ref)

print(f"batch1 run_id: {r1.context.run_id}")
print(f"batch2 run_id: {r2.context.run_id}")
print(f"IDs distinct : {r1.context.run_id != r2.context.run_id}")

# Profile means differ between batches
print(f"\nbatch1 age mean : {r1.profile['age']['mean']:.2f}")
print(f"batch2 age mean : {r2.profile['age']['mean']:.2f}")

batch1 run_id: b89d8ab72b75450b81d9960241ae716e
batch2 run_id: 2c4eefdf9593428ba4d6c426aa54332a
IDs distinct : True

batch1 age mean : 48.08
batch2 age mean : 46.03


## 5. Profiler standalone

`profile_columns()` can be called directly, without going through the Runner.  
Useful for profiling ad hoc DataFrames or comparing reference vs current.

In [11]:
from ayn_ml.metrics.tabular.profiler import profile_columns

cols_to_profile = ["age", "income", "region", "gender"]
prof_ref = profile_columns(df_ref, cols_to_profile, schema)
prof_cur = profile_columns(df_cur, cols_to_profile, schema)

# Drift summary — delta on numeric stats
print(f"{'column':<10} {'stat':<8} {'reference':>12} {'current':>12} {'delta':>10}")
print("-" * 56)
for col in ["age", "income"]:
    for stat in ["mean", "std", "p50"]:
        ref_v = prof_ref[col][stat]
        cur_v = prof_cur[col][stat]
        delta = cur_v - ref_v
        print(f"{col:<10} {stat:<8} {ref_v:>12.2f} {cur_v:>12.2f} {delta:>+10.2f}")

print()
# Categorical summary
for col in ["region", "gender"]:
    print(
        f"{col}: ref n_unique={prof_ref[col]['n_unique']}  cur n_unique={prof_cur[col]['n_unique']}"
        f"  ref top={prof_ref[col]['top_category']}  cur top={prof_cur[col]['top_category']}"
    )

column     stat        reference      current      delta
--------------------------------------------------------
age        mean            40.17        47.05      +6.88
age        std              8.08        10.04      +1.96
age        p50             40.25        46.83      +6.58
income     mean         52876.00     56059.40   +3183.40
income     std          12123.56     13196.02   +1072.46
income     p50          53193.41     55611.06   +2417.65

region: ref n_unique=4  cur n_unique=5  ref top=west  cur top=south
gender: ref n_unique=2  cur n_unique=2  ref top=M  cur top=M
